# Train your own Piper voice (for Voice Keyer)

This notebook fine-tunes a Piper text-to-speech voice on **your recordings** and gives you back a
`.onnx` voice you can use in Voice Keyer.

**Before you start**
1. In Voice Keyer, open the gear menu > *Voice trainer*, record at least 150 prompts (250+ is better), then click *Export for training*. That makes `<name>_dataset.zip`.
2. In Colab choose **Runtime > Change runtime type > T4 GPU** (or better).
3. Run the cells below from top to bottom. Training takes roughly 1 to 3 hours depending on the settings in step 4.

**When it finishes** it downloads `<name>_voice.zip`. Unzip it, then in Voice Keyer open *Settings > Import voice...* and pick the `.onnx` file (keep the `.onnx.json` next to it).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Piper training (takes a few minutes)

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y build-essential cmake ninja-build espeak-ng > /dev/null
cd /content
[ -d piper1-gpl ] || git clone --depth 1 https://github.com/OHF-voice/piper1-gpl.git
cd piper1-gpl
python3 -m pip install -q scikit-build cmake ninja setuptools wheel cython numpy
python3 -m pip install -q -e '.[train]'
python3 setup.py build_ext --inplace
bash build_monotonic_align.sh
ls src/piper/train/vits/monotonic_align/monotonic_align/*.so
echo INSTALL OK

## 3. Settings

* `VOICE_NAME` - letters, digits and underscores only. Becomes the voice's file name.
* `EXTRA_EPOCHS` - how long to fine-tune. 1000 is a good first try for ~250 clips; add more if it still sounds rough.
* `BATCH_SIZE` - lower this (8) if you get an out-of-memory error.
* `SAVE_TO_DRIVE` - keeps checkpoints in your Google Drive so you can resume if Colab disconnects.

In [ ]:
VOICE_NAME = "MyVoice"
EXTRA_EPOCHS = 1000
BATCH_SIZE = 16
SAVE_TO_DRIVE = True

BASE_EPOCH = 2164  # epoch of the base checkpoint below; training must run past it
BASE_CKPT_URL = ("https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/"
                 "en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt")
MAX_EPOCHS = BASE_EPOCH + EXTRA_EPOCHS

## 4. Upload your dataset zip

In [ ]:
import os, re, zipfile
from google.colab import files

assert re.fullmatch(r"[A-Za-z0-9_]+", VOICE_NAME), "VOICE_NAME: letters, digits, underscores only"
os.makedirs("/content/dataset", exist_ok=True)
print("Choose the *_dataset.zip exported by Voice Keyer...")
uploaded = files.upload()
for name in uploaded:
    zipfile.ZipFile(name).extractall("/content/dataset")
clips = [f for f in os.listdir("/content/dataset/wavs") if f.endswith(".wav")]
lines = open("/content/dataset/metadata.csv", encoding="utf-8").read().strip().splitlines()
print(f"{len(clips)} wav files, {len(lines)} metadata lines")
assert len(clips) >= 50, "Too few clips - record more prompts first"

## 5. Download the base voice to fine-tune from

In [ ]:
import subprocess
if not os.path.exists("/content/base.ckpt"):
    subprocess.run(["wget", "-q", "--show-progress", "-O", "/content/base.ckpt", BASE_CKPT_URL], check=True)
print("base checkpoint:", os.path.getsize("/content/base.ckpt") // 1_000_000, "MB")

## 6. Where to keep checkpoints (and resume if a previous run exists)

In [ ]:
import glob
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    TRAIN_DIR = f"/content/drive/MyDrive/piper_training/{VOICE_NAME}"
else:
    TRAIN_DIR = f"/content/training/{VOICE_NAME}"
os.makedirs(TRAIN_DIR, exist_ok=True)

import zipfile
def _readable(f):
    # A checkpoint cut off mid-save (e.g. when you press Stop) is not a valid zip; skip it.
    try:
        with zipfile.ZipFile(f) as z: return z.testzip() is None
    except Exception:
        return False

def latest_checkpoint():
    found = glob.glob(f"{TRAIN_DIR}/**/*.ckpt", recursive=True)
    found = [f for f in found if _readable(f)]
    return max(found, key=os.path.getmtime) if found else None

CKPT = latest_checkpoint() or "/content/base.ckpt"
print("starting from:", CKPT)

## 7. Train

First a short step makes the base voice loadable by this version of Piper (it is older, and PyTorch 2.6+ is stricter). Then the long step trains. Lines like `Epoch 2300: ...` show progress. If Colab disconnects, reconnect, run
cells 2-6 again (the dataset upload is needed again) and re-run these two cells: training resumes from your latest checkpoint.

In [ ]:
# Make the checkpoint loadable: pathlib paths become plain strings, and settings this Piper no longer accepts
# (sample_bytes and others) are dropped. The base file comes from Piper's own Hugging Face repo.
import inspect, pathlib, sys, torch
sys.path.insert(0, "/content/piper1-gpl/src")
from piper.train.vits.lightning import VitsModel

def _plain(o):
    if isinstance(o, pathlib.PurePath): return str(o)
    if isinstance(o, dict): return {k: _plain(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return type(o)(_plain(v) for v in o)
    return o

_ck = _plain(torch.load(CKPT, weights_only=False, map_location="cpu"))
_ok = set(inspect.signature(VitsModel.__init__).parameters)
_hp = _ck.get("hyper_parameters", {})
for k in [k for k in _hp if k not in _ok]:
    del _hp[k]
torch.save(_ck, CKPT)
del _ck
print("checkpoint ready:", CKPT)

In [ ]:
%cd /content/piper1-gpl
!python3 -m piper.train fit   --data.voice_name "{VOICE_NAME}"   --data.csv_path /content/dataset/metadata.csv   --data.audio_dir /content/dataset/wavs/   --model.sample_rate 22050   --data.espeak_voice "en-us"   --data.cache_dir /content/cache/   --data.config_path /content/config.json   --data.batch_size {BATCH_SIZE}   --trainer.max_epochs {MAX_EPOCHS}   --trainer.default_root_dir "{TRAIN_DIR}"   --ckpt_path "{CKPT}"

## 8. Export the voice and download it

In [ ]:
import shutil
subprocess.run(["python3", "-m", "pip", "install", "-q", "onnx", "onnxscript", "onnxruntime"], check=True)  # the exporter needs these
ckpt = latest_checkpoint()
assert ckpt, "No checkpoint found - did training run?"
print("exporting", ckpt)
os.makedirs("/content/out", exist_ok=True)
onnx = f"/content/out/{VOICE_NAME}.onnx"
# Run the exporter with PyTorch's strict loading switched off (the checkpoint is your own file), and show its messages.
runner = ("import torch, runpy; _l = torch.load; "
          "torch.load = lambda *a, **k: _l(*a, **{**k, 'weights_only': False}); _e = torch.onnx.export; torch.onnx.export = lambda *a, **k: _e(*a, **{**k, 'dynamo': False}); "
          "runpy.run_module('piper.train.export_onnx', run_name='__main__')")
r = subprocess.run(["python3", "-c", runner, "--checkpoint", ckpt, "--output-file", onnx], cwd="/content/piper1-gpl", capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
assert r.returncode == 0, "Export failed - copy the messages above and send them to Claude"
shutil.copy("/content/config.json", onnx + ".json")
archive = shutil.make_archive(f"/content/{VOICE_NAME}_voice", "zip", "/content/out")
files.download(archive)

## 9. (Optional) Listen to it here

If this cell errors the voice is probably still fine - test it in Voice Keyer instead (Settings > Import voice, then right-click a button > Preview).

In [ ]:
from IPython.display import Audio, display
!python3 -m piper -m /content/out/{VOICE_NAME}.onnx -f /content/test.wav -- "CQ contest, this is Whiskey Two Romeo calling CQ."
display(Audio("/content/test.wav"))